# 02 — Data Cleaning & Quality Audit

**Sector:** Aviation / Transportation  
**Project:** US Flight Ops Dashboard (Capstone 2)  
**Objective:** Build a reproducible, documented ETL pipeline targeting operational efficiency and delay reduction.

---

## 2.1 Import Libraries

In [31]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2.2 Auto-Detect Paths & Load Raw Data

In [32]:
# Auto-detect project root and data paths
def find_data_path():
    """Auto-detect the location of raw CSV files."""
    candidates = [
        '../data/raw/',          # Running from notebooks/ folder
        'data/raw/',             # Running from project root
        './',                    # CSVs in current directory
        '../',                   # One level up
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'airlines.csv')):
            print(f"✅ Data found at: {os.path.abspath(path)}")
            return path
    raise FileNotFoundError("Could not find airlines.csv. Please check your file locations.")

RAW_DATA_PATH = find_data_path()

# Set processed output path
if RAW_DATA_PATH.startswith('../'):
    PROCESSED_DATA_PATH = '../data/processed/'
elif RAW_DATA_PATH.startswith('data/'):
    PROCESSED_DATA_PATH = 'data/processed/'
else:
    PROCESSED_DATA_PATH = 'data/processed/'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Load all datasets
print("\nLoading airlines...")
df_airlines = pd.read_csv(os.path.join(RAW_DATA_PATH, 'airlines.csv'))
print(f"  ✅ Airlines: {df_airlines.shape}")

print("Loading airports...")
df_airports = pd.read_csv(os.path.join(RAW_DATA_PATH, 'airports.csv'))
print(f"  ✅ Airports: {df_airports.shape}")

print("Loading flights (this may take a moment)...")
df_flights = pd.read_csv(os.path.join(RAW_DATA_PATH, 'flights.csv'), low_memory=False)
print(f"  ✅ Flights: {df_flights.shape}")

print(f"\nTotal raw records: {len(df_flights):,}")

✅ Data found at: /Users/yashraghubanshi/Downloads/dva/data/raw

Loading airlines...
  ✅ Airlines: (14, 2)
Loading airports...
  ✅ Airports: (322, 7)
Loading flights (this may take a moment)...
  ✅ Flights: (100000, 31)

Total raw records: 100,000


## 2.3 Pre-Cleaning Snapshot

Capture the state of the data *before* any transformations for audit trail.

In [33]:
# Record pre-cleaning state
pre_clean_shape = df_flights.shape
pre_clean_missing = df_flights.isnull().sum()
pre_clean_dtypes = df_flights.dtypes.copy()

print(f"Pre-cleaning shape: {pre_clean_shape}")
print(f"\nPre-cleaning missing values:")
for col in df_flights.columns:
    miss = df_flights[col].isnull().sum()
    if miss > 0:
        pct = miss / len(df_flights) * 100
        print(f"  {col:30s} | {miss:>10,} ({pct:.2f}%)")

Pre-cleaning shape: (100000, 31)

Pre-cleaning missing values:
  TAIL_NUMBER                    |        260 (0.26%)
  DEPARTURE_TIME                 |      1,481 (1.48%)
  DEPARTURE_DELAY                |      1,481 (1.48%)
  TAXI_OUT                       |      1,538 (1.54%)
  WHEELS_OFF                     |      1,538 (1.54%)
  ELAPSED_TIME                   |      1,805 (1.80%)
  AIR_TIME                       |      1,805 (1.80%)
  WHEELS_ON                      |      1,602 (1.60%)
  TAXI_IN                        |      1,602 (1.60%)
  ARRIVAL_TIME                   |      1,602 (1.60%)
  ARRIVAL_DELAY                  |      1,805 (1.80%)
  CANCELLATION_REASON            |     98,446 (98.45%)
  AIR_SYSTEM_DELAY               |     81,646 (81.65%)
  SECURITY_DELAY                 |     81,646 (81.65%)
  AIRLINE_DELAY                  |     81,646 (81.65%)
  LATE_AIRCRAFT_DELAY            |     81,646 (81.65%)
  WEATHER_DELAY                  |     81,646 (81.65%)


## 2.4 Step 1 — Handle Missing Values

**Strategy:**
- **Delay breakdown columns** (AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, WEATHER_DELAY): Fill with 0 — missing means no delay of that type.
- **CANCELLATION_REASON**: Fill with 'N' (Not Cancelled) — missing means the flight was not cancelled.
- **DEPARTURE_TIME, DEPARTURE_DELAY, TAXI_OUT, WHEELS_OFF, ELAPSED_TIME, AIR_TIME, WHEELS_ON, TAXI_IN, ARRIVAL_TIME, ARRIVAL_DELAY**: These are missing for cancelled flights. We'll keep NaN but flag them.
- **TAIL_NUMBER**: Fill missing with 'UNKNOWN'.

In [34]:
# --- Step 1a: Fill delay breakdown columns with 0 ---
delay_columns = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 
                 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

for col in delay_columns:
    before = df_flights[col].isnull().sum()
    df_flights[col] = df_flights[col].fillna(0)
    after = df_flights[col].isnull().sum()
    print(f"  {col}: {before:,} nulls → {after:,} nulls (filled with 0)")

print("\n✅ Delay breakdown columns cleaned.")

  AIR_SYSTEM_DELAY: 81,646 nulls → 0 nulls (filled with 0)
  SECURITY_DELAY: 81,646 nulls → 0 nulls (filled with 0)
  AIRLINE_DELAY: 81,646 nulls → 0 nulls (filled with 0)
  LATE_AIRCRAFT_DELAY: 81,646 nulls → 0 nulls (filled with 0)
  WEATHER_DELAY: 81,646 nulls → 0 nulls (filled with 0)

✅ Delay breakdown columns cleaned.


In [35]:
# --- Step 1b: Fill CANCELLATION_REASON ---
before = df_flights['CANCELLATION_REASON'].isnull().sum()
df_flights['CANCELLATION_REASON'] = df_flights['CANCELLATION_REASON'].fillna('N')
print(f"  CANCELLATION_REASON: {before:,} nulls → {df_flights['CANCELLATION_REASON'].isnull().sum():,} nulls")
print(f"  Value counts: {dict(df_flights['CANCELLATION_REASON'].value_counts())}")

# Map cancellation codes to readable labels
cancellation_map = {
    'A': 'Airline/Carrier',
    'B': 'Weather',
    'C': 'National Air System',
    'D': 'Security',
    'N': 'Not Cancelled'
}
df_flights['CANCELLATION_REASON_DESC'] = df_flights['CANCELLATION_REASON'].map(cancellation_map)
print(f"\n✅ Cancellation reason cleaned and mapped to descriptions.")

  CANCELLATION_REASON: 98,446 nulls → 0 nulls
  Value counts: {'N': np.int64(98446), 'B': np.int64(809), 'A': np.int64(443), 'C': np.int64(302)}

✅ Cancellation reason cleaned and mapped to descriptions.


In [36]:
# --- Step 1c: Fill TAIL_NUMBER ---
before = df_flights['TAIL_NUMBER'].isnull().sum()
df_flights['TAIL_NUMBER'] = df_flights['TAIL_NUMBER'].fillna('UNKNOWN')
print(f"  TAIL_NUMBER: {before:,} nulls → filled with 'UNKNOWN'")

print("\n✅ Missing value handling complete.")

  TAIL_NUMBER: 260 nulls → filled with 'UNKNOWN'

✅ Missing value handling complete.


## 2.5 Step 2 — Data Type Conversions

**Key transformations:**
- Convert SCHEDULED_DEPARTURE, DEPARTURE_TIME, SCHEDULED_ARRIVAL, ARRIVAL_TIME from integer HHMM format to proper time strings
- Create a proper DATE column from YEAR, MONTH, DAY
- Convert categorical columns to category dtype for memory efficiency

In [37]:
# --- Step 2a: Create proper DATE column ---
df_flights['DATE'] = pd.to_datetime(
    df_flights[['YEAR', 'MONTH', 'DAY']].assign(
        YEAR=df_flights['YEAR'].astype(str),
        MONTH=df_flights['MONTH'].astype(str).str.zfill(2),
        DAY=df_flights['DAY'].astype(str).str.zfill(2)
    ).agg('-'.join, axis=1),
    format='%Y-%m-%d',
    errors='coerce'
)

print(f"Date range: {df_flights['DATE'].min()} to {df_flights['DATE'].max()}")
print(f"Null dates: {df_flights['DATE'].isnull().sum()}")
print(f"\n✅ DATE column created.")

Date range: 2015-01-01 00:00:00 to 2015-12-31 00:00:00
Null dates: 0

✅ DATE column created.


In [38]:
# --- Step 2b: Convert time columns from HHMM integer to HH:MM string ---
def convert_hhmm(val):
    """Convert HHMM integer (e.g., 1430) to 'HH:MM' string."""
    if pd.isna(val):
        return np.nan
    val = int(val)
    # Handle 2400 edge case (midnight)
    if val == 2400:
        val = 0
    hours = val // 100
    minutes = val % 100
    return f"{hours:02d}:{minutes:02d}"

time_columns = ['SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL']

for col in time_columns:
    new_col = col + '_TIME'
    df_flights[new_col] = df_flights[col].apply(convert_hhmm)
    print(f"  Created {new_col} from {col}")
    print(f"    Sample: {df_flights[col].head(3).tolist()} → {df_flights[new_col].head(3).tolist()}")

print(f"\n✅ Time columns converted.")

  Created SCHEDULED_DEPARTURE_TIME from SCHEDULED_DEPARTURE
    Sample: [1340, 1910, 630] → ['13:40', '19:10', '06:30']
  Created SCHEDULED_ARRIVAL_TIME from SCHEDULED_ARRIVAL
    Sample: [1436, 2145, 820] → ['14:36', '21:45', '08:20']

✅ Time columns converted.


In [39]:
# --- Step 2c: Extract time-based features ---
# Month name
month_map = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
             7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
df_flights['MONTH_NAME'] = df_flights['MONTH'].map(month_map)

# Day of week name
day_map = {1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 
           5: 'Friday', 6: 'Saturday', 7: 'Sunday'}
df_flights['DAY_NAME'] = df_flights['DAY_OF_WEEK'].map(day_map)

# Quarter
df_flights['QUARTER'] = df_flights['MONTH'].apply(lambda x: f"Q{(x-1)//3 + 1}")

# Departure hour bucket
df_flights['DEP_HOUR'] = (df_flights['SCHEDULED_DEPARTURE'] // 100).astype('Int64')

# Time of day category
def get_time_of_day(hour):
    if pd.isna(hour):
        return 'Unknown'
    hour = int(hour)
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_flights['TIME_OF_DAY'] = df_flights['DEP_HOUR'].apply(get_time_of_day)

print("New time features created:")
print(f"  MONTH_NAME: {df_flights['MONTH_NAME'].unique()[:4]}...")
print(f"  DAY_NAME: {df_flights['DAY_NAME'].unique()[:4]}...")
print(f"  QUARTER: {df_flights['QUARTER'].unique()}")
print(f"  DEP_HOUR range: {df_flights['DEP_HOUR'].min()} to {df_flights['DEP_HOUR'].max()}")
print(f"  TIME_OF_DAY: {df_flights['TIME_OF_DAY'].value_counts().to_dict()}")
print(f"\n✅ Time-based features engineered.")

New time features created:
  MONTH_NAME: <StringArray>
['April', 'January', 'July', 'May']
Length: 4, dtype: str...
  DAY_NAME: <StringArray>
['Tuesday', 'Saturday', 'Wednesday', 'Monday']
Length: 4, dtype: str...
  QUARTER: <StringArray>
['Q2', 'Q1', 'Q3', 'Q4']
Length: 4, dtype: str
  DEP_HOUR range: 0 to 23
  TIME_OF_DAY: {'Morning': 40990, 'Afternoon': 30142, 'Evening': 22541, 'Night': 6327}

✅ Time-based features engineered.


In [40]:
# --- Step 2d: Convert categorical columns to category dtype ---
cat_columns = ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'CANCELLATION_REASON',
               'CANCELLATION_REASON_DESC', 'MONTH_NAME', 'DAY_NAME', 'QUARTER', 'TIME_OF_DAY']

for col in cat_columns:
    df_flights[col] = df_flights[col].astype('category')
    print(f"  {col} → category dtype ({df_flights[col].cat.categories.size} categories)")

print(f"\nMemory usage after dtype optimization: {df_flights.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"\n✅ Categorical columns optimized.")

  AIRLINE → category dtype (14 categories)
  ORIGIN_AIRPORT → category dtype (588 categories)
  DESTINATION_AIRPORT → category dtype (590 categories)
  CANCELLATION_REASON → category dtype (4 categories)
  CANCELLATION_REASON_DESC → category dtype (4 categories)
  MONTH_NAME → category dtype (12 categories)
  DAY_NAME → category dtype (7 categories)
  QUARTER → category dtype (4 categories)
  TIME_OF_DAY → category dtype (4 categories)

Memory usage after dtype optimization: 38.11 MB

✅ Categorical columns optimized.


## 2.6 Step 3 — Outlier Detection & Treatment

**Approach:** Use IQR method on key numerical columns. Flag extreme outliers but do not remove them — delays can legitimately be very high.

In [41]:
# --- Step 3a: Outlier detection using IQR ---
numerical_cols_for_outliers = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'ELAPSED_TIME', 
                                'AIR_TIME', 'DISTANCE', 'TAXI_OUT', 'TAXI_IN']

outlier_report = []
for col in numerical_cols_for_outliers:
    data = df_flights[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((data < lower) | (data > upper)).sum()
    pct = outliers / len(data) * 100
    outlier_report.append({
        'Column': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'Lower': lower, 'Upper': upper,
        'Outliers': outliers, 'Outlier_%': round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_report)
print("Outlier Detection Report (IQR Method):")
outlier_df

Outlier Detection Report (IQR Method):


,Column,Q1,Q3,IQR,Lower,Upper,Outliers,Outlier_%
0,DEPARTURE_DELAY,-5.0,7.0,12.0,-23.0,25.0,12696,12.89
1,ARRIVAL_DELAY,-13.0,8.0,21.0,-44.5,39.5,8880,9.04
2,ELAPSED_TIME,83.0,169.0,86.0,-46.0,298.0,4942,5.03
3,AIR_TIME,61.0,144.0,83.0,-63.5,268.5,5257,5.35
4,DISTANCE,373.0,1065.0,692.0,-665.0,2103.0,6035,6.04
5,TAXI_OUT,11.0,19.0,8.0,-1.0,31.0,4810,4.89
6,TAXI_IN,4.0,9.0,5.0,-3.5,16.5,5030,5.11


In [42]:
# --- Step 3b: Flag outlier records (don't remove — delays can be legitimately extreme) ---
# Flag flights with extreme departure delay (> 180 minutes = 3 hours)
df_flights['IS_EXTREME_DEP_DELAY'] = (df_flights['DEPARTURE_DELAY'] > 180).astype(int)
df_flights['IS_EXTREME_ARR_DELAY'] = (df_flights['ARRIVAL_DELAY'] > 180).astype(int)

print(f"Flights with extreme departure delay (>3hrs): {df_flights['IS_EXTREME_DEP_DELAY'].sum():,} ({df_flights['IS_EXTREME_DEP_DELAY'].mean()*100:.2f}%)")
print(f"Flights with extreme arrival delay (>3hrs):   {df_flights['IS_EXTREME_ARR_DELAY'].sum():,} ({df_flights['IS_EXTREME_ARR_DELAY'].mean()*100:.2f}%)")

# Cap extreme values for delay columns to prevent skewing (winsorize at 99.5th percentile)
# We keep original columns and create capped versions
for col in ['DEPARTURE_DELAY', 'ARRIVAL_DELAY']:
    cap_val = df_flights[col].quantile(0.995)
    capped_col = col + '_CAPPED'
    df_flights[capped_col] = df_flights[col].clip(upper=cap_val)
    print(f"  {col} capped at {cap_val:.0f} min → {capped_col}")

print(f"\n✅ Outlier flags and capped columns created.")

Flights with extreme departure delay (>3hrs): 791 (0.79%)
Flights with extreme arrival delay (>3hrs):   780 (0.78%)
  DEPARTURE_DELAY capped at 219 min → DEPARTURE_DELAY_CAPPED
  ARRIVAL_DELAY capped at 216 min → ARRIVAL_DELAY_CAPPED

✅ Outlier flags and capped columns created.


## 2.7 Step 4 — Feature Engineering

Create derived columns that add analytical value.

In [43]:
# --- Step 4a: Delay classification ---
def classify_delay(delay):
    """Classify delay into categories based on minutes."""
    if pd.isna(delay):
        return 'Cancelled'
    elif delay <= 0:
        return 'On Time / Early'
    elif delay <= 15:
        return 'Minor Delay (1-15 min)'
    elif delay <= 60:
        return 'Moderate Delay (16-60 min)'
    elif delay <= 180:
        return 'Significant Delay (1-3 hrs)'
    else:
        return 'Severe Delay (3+ hrs)'

df_flights['DELAY_CATEGORY'] = df_flights['ARRIVAL_DELAY'].apply(classify_delay)

print("Delay Category Distribution:")
print(df_flights['DELAY_CATEGORY'].value_counts())
print(f"\n✅ DELAY_CATEGORY column created.")

Delay Category Distribution:
DELAY_CATEGORY
On Time / Early                62197
Minor Delay (1-15 min)         18306
Moderate Delay (16-60 min)     12167
Significant Delay (1-3 hrs)     4745
Cancelled                       1805
Severe Delay (3+ hrs)            780
Name: count, dtype: int64

✅ DELAY_CATEGORY column created.


In [44]:
# --- Step 4b: Is Delayed flag (>15 min is industry standard for 'delayed') ---
df_flights['IS_DELAYED'] = (df_flights['ARRIVAL_DELAY'] > 15).astype('Int64')

# Handle NaN (cancelled flights) — they are not counted as delayed
df_flights.loc[df_flights['ARRIVAL_DELAY'].isna(), 'IS_DELAYED'] = pd.NA

print(f"Delayed flights (>15 min): {df_flights['IS_DELAYED'].sum():,}")
non_cancelled = df_flights[df_flights['CANCELLED'] == 0]
print(f"Delay rate (excl. cancelled): {non_cancelled['IS_DELAYED'].mean()*100:.2f}%")
print(f"\n✅ IS_DELAYED flag created.")

Delayed flights (>15 min): 17,692
Delay rate (excl. cancelled): 18.02%

✅ IS_DELAYED flag created.


In [45]:
# --- Step 4c: Total delay breakdown ---
df_flights['TOTAL_DELAY_BREAKDOWN'] = (
    df_flights['AIR_SYSTEM_DELAY'] + 
    df_flights['SECURITY_DELAY'] + 
    df_flights['AIRLINE_DELAY'] + 
    df_flights['LATE_AIRCRAFT_DELAY'] + 
    df_flights['WEATHER_DELAY']
)

# Primary delay cause
delay_type_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 
                   'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

df_flights['PRIMARY_DELAY_CAUSE'] = df_flights[delay_type_cols].idxmax(axis=1)
# Only assign for delayed flights
df_flights.loc[df_flights['TOTAL_DELAY_BREAKDOWN'] == 0, 'PRIMARY_DELAY_CAUSE'] = 'No Delay'

# Clean up the names
cause_rename = {
    'AIR_SYSTEM_DELAY': 'Air System',
    'SECURITY_DELAY': 'Security',
    'AIRLINE_DELAY': 'Airline/Carrier',
    'LATE_AIRCRAFT_DELAY': 'Late Aircraft',
    'WEATHER_DELAY': 'Weather',
    'No Delay': 'No Delay'
}
df_flights['PRIMARY_DELAY_CAUSE'] = df_flights['PRIMARY_DELAY_CAUSE'].map(cause_rename)

print("Primary Delay Cause Distribution:")
print(df_flights['PRIMARY_DELAY_CAUSE'].value_counts())
print(f"\n✅ Delay breakdown and primary cause columns created.")

Primary Delay Cause Distribution:
PRIMARY_DELAY_CAUSE
No Delay           81646
Late Aircraft       7056
Air System          5318
Airline/Carrier     5316
Weather              628
Security              36
Name: count, dtype: int64

✅ Delay breakdown and primary cause columns created.


In [46]:
# --- Step 4d: Distance categories ---
def classify_distance(dist):
    if pd.isna(dist):
        return 'Unknown'
    elif dist <= 500:
        return 'Short-haul (<=500 mi)'
    elif dist <= 1500:
        return 'Medium-haul (501-1500 mi)'
    else:
        return 'Long-haul (>1500 mi)'

df_flights['DISTANCE_CATEGORY'] = df_flights['DISTANCE'].apply(classify_distance)

print("Distance Category Distribution:")
print(df_flights['DISTANCE_CATEGORY'].value_counts())
print(f"\n✅ DISTANCE_CATEGORY column created.")

Distance Category Distribution:
DISTANCE_CATEGORY
Medium-haul (501-1500 mi)    49944
Short-haul (<=500 mi)        36573
Long-haul (>1500 mi)         13483
Name: count, dtype: int64

✅ DISTANCE_CATEGORY column created.


In [47]:
# --- Step 4e: Route column ---
df_flights['ROUTE'] = df_flights['ORIGIN_AIRPORT'].astype(str) + ' -> ' + df_flights['DESTINATION_AIRPORT'].astype(str)

print(f"Unique routes: {df_flights['ROUTE'].nunique():,}")
print(f"\nTop 10 busiest routes:")
print(df_flights['ROUTE'].value_counts().head(10))
print(f"\n✅ ROUTE column created.")

Unique routes: 6,980

Top 10 busiest routes:
ROUTE
LAX -> SFO    240
SFO -> LAX    230
JFK -> LAX    204
LAX -> JFK    203
LAS -> LAX    184
JFK -> SFO    171
ORD -> LGA    168
ATL -> MCO    156
LGA -> ORD    152
LAX -> ORD    145
Name: count, dtype: int64

✅ ROUTE column created.


## 2.8 Step 5 — Merge Reference Tables

Enrich the flights data with airline names and airport details.

In [48]:
# --- Step 5a: Merge airline names ---
df_flights = df_flights.merge(
    df_airlines.rename(columns={'IATA_CODE': 'AIRLINE', 'AIRLINE': 'AIRLINE_NAME'}),
    on='AIRLINE',
    how='left'
)

print(f"Airline name merge — null AIRLINE_NAME: {df_flights['AIRLINE_NAME'].isnull().sum()}")
print(f"\nAirline mapping:")
for _, row in df_airlines.iterrows():
    print(f"  {row['IATA_CODE']} -> {row['AIRLINE']}")

Airline name merge — null AIRLINE_NAME: 0

Airline mapping:
  UA -> United Air Lines Inc.
  AA -> American Airlines Inc.
  US -> US Airways Inc.
  F9 -> Frontier Airlines Inc.
  B6 -> JetBlue Airways
  OO -> Skywest Airlines Inc.
  AS -> Alaska Airlines Inc.
  NK -> Spirit Air Lines
  WN -> Southwest Airlines Co.
  DL -> Delta Air Lines Inc.
  EV -> Atlantic Southeast Airlines
  HA -> Hawaiian Airlines Inc.
  MQ -> American Eagle Airlines Inc.
  VX -> Virgin America


In [49]:
# --- Step 5b: Merge origin airport details ---
origin_cols = df_airports.rename(columns={
    'IATA_CODE': 'ORIGIN_AIRPORT',
    'AIRPORT': 'ORIGIN_AIRPORT_NAME',
    'CITY': 'ORIGIN_CITY',
    'STATE': 'ORIGIN_STATE',
    'LATITUDE': 'ORIGIN_LAT',
    'LONGITUDE': 'ORIGIN_LONG'
})[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE', 'ORIGIN_LAT', 'ORIGIN_LONG']]

df_flights = df_flights.merge(origin_cols, on='ORIGIN_AIRPORT', how='left')

# --- Step 5c: Merge destination airport details ---
dest_cols = df_airports.rename(columns={
    'IATA_CODE': 'DESTINATION_AIRPORT',
    'AIRPORT': 'DEST_AIRPORT_NAME',
    'CITY': 'DEST_CITY',
    'STATE': 'DEST_STATE',
    'LATITUDE': 'DEST_LAT',
    'LONGITUDE': 'DEST_LONG'
})[['DESTINATION_AIRPORT', 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'DEST_LAT', 'DEST_LONG']]

df_flights = df_flights.merge(dest_cols, on='DESTINATION_AIRPORT', how='left')

print(f"Origin merges — null ORIGIN_CITY: {df_flights['ORIGIN_CITY'].isnull().sum()}")
print(f"Dest merges   — null DEST_CITY: {df_flights['DEST_CITY'].isnull().sum()}")
print(f"\n✅ Airport details merged for origin and destination.")

Origin merges — null ORIGIN_CITY: 8203
Dest merges   — null DEST_CITY: 8203

✅ Airport details merged for origin and destination.


## 2.9 Step 6 — Post-Cleaning Validation

In [50]:
# Post-cleaning summary
print("=" * 80)
print("POST-CLEANING DATA SUMMARY")
print("=" * 80)
print(f"\nShape: {df_flights.shape} (was {pre_clean_shape})")
print(f"New columns added: {df_flights.shape[1] - pre_clean_shape[1]}")
print(f"\nMemory usage: {df_flights.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\n--- All Columns ({len(df_flights.columns)}) ---")
for i, col in enumerate(df_flights.columns):
    print(f"  {i+1:2d}. {col:35s} | {str(df_flights[col].dtype):15s} | Nulls: {df_flights[col].isnull().sum():>10,}")

POST-CLEANING DATA SUMMARY

Shape: (100000, 61) (was (100000, 31))
New columns added: 30

Memory usage: 127.06 MB

--- All Columns (61) ---
   1. YEAR                                | int64           | Nulls:          0
   2. MONTH                               | int64           | Nulls:          0
   3. DAY                                 | int64           | Nulls:          0
   4. DAY_OF_WEEK                         | int64           | Nulls:          0
   5. AIRLINE                             | str             | Nulls:          0
   6. FLIGHT_NUMBER                       | int64           | Nulls:          0
   7. TAIL_NUMBER                         | str             | Nulls:          0
   8. ORIGIN_AIRPORT                      | str             | Nulls:          0
   9. DESTINATION_AIRPORT                 | str             | Nulls:          0
  10. SCHEDULED_DEPARTURE                 | int64           | Nulls:          0
  11. DEPARTURE_TIME                      | float64         

In [51]:
# Sanity checks
print("\nSanity Checks:")
print(f"  Cancelled flights: {df_flights['CANCELLED'].sum():,}")
print(f"  Diverted flights: {df_flights['DIVERTED'].sum():,}")
print(f"  Flights with arrival delay: {(df_flights['ARRIVAL_DELAY'] > 0).sum():,}")
print(f"  Flights on time or early: {(df_flights['ARRIVAL_DELAY'] <= 0).sum():,}")
print(f"  Unique airlines: {df_flights['AIRLINE'].nunique()}")
print(f"  Unique routes: {df_flights['ROUTE'].nunique():,}")
print(f"  Date range: {df_flights['DATE'].min()} to {df_flights['DATE'].max()}")
print(f"  Months covered: {sorted(df_flights['MONTH'].unique())}")

# Verify no data loss
assert len(df_flights) == pre_clean_shape[0], "Row count mismatch!"
print(f"\n✅ All sanity checks passed. No data loss.")


Sanity Checks:
  Cancelled flights: 1,554
  Diverted flights: 251
  Flights with arrival delay: 35,998
  Flights on time or early: 62,197
  Unique airlines: 14
  Unique routes: 6,980
  Date range: 2015-01-01 00:00:00 to 2015-12-31 00:00:00
  Months covered: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

✅ All sanity checks passed. No data loss.


## 2.10 Step 7 — Export Cleaned Data

In [52]:
# Export cleaned flights dataset
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
output_path = os.path.join(PROCESSED_DATA_PATH, 'flights_cleaned.csv')
print(f"Exporting cleaned dataset to: {output_path}")
print(f"Shape: {df_flights.shape}")

df_flights.to_csv(output_path, index=False)
file_size = os.path.getsize(output_path) / (1024**2)
print(f"\n✅ Cleaned dataset saved: {file_size:.2f} MB")

Exporting cleaned dataset to: ../data/processed/flights_cleaned.csv
Shape: (100000, 61)

✅ Cleaned dataset saved: 41.88 MB


In [53]:
# Export cleaned reference tables
df_airlines.to_csv(os.path.join(PROCESSED_DATA_PATH, 'airlines_cleaned.csv'), index=False)
df_airports.to_csv(os.path.join(PROCESSED_DATA_PATH, 'airports_cleaned.csv'), index=False)

print("✅ Reference tables exported.")
print(f"\n--- Processed Files ---")
for f in os.listdir(PROCESSED_DATA_PATH):
    fpath = os.path.join(PROCESSED_DATA_PATH, f)
    if os.path.isfile(fpath):
        print(f"  {f}: {os.path.getsize(fpath) / (1024**2):.2f} MB")

✅ Reference tables exported.

--- Processed Files ---
  airports_cleaned.csv: 0.02 MB
  flights_cleaned.csv: 41.88 MB
  airlines_cleaned.csv: 0.00 MB


## 2.11 Cleaning Log Summary

| Step | Action | Details |
|------|--------|---------|
| 1a | Fill delay columns | AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, WEATHER_DELAY -> filled with 0 |
| 1b | Fill CANCELLATION_REASON | NaN -> 'N' (Not Cancelled), mapped to descriptive labels |
| 1c | Fill TAIL_NUMBER | NaN -> 'UNKNOWN' |
| 2a | Create DATE | Combined YEAR, MONTH, DAY -> proper datetime |
| 2b | Convert times | SCHEDULED_DEPARTURE, SCHEDULED_ARRIVAL -> HH:MM format |
| 2c | Time features | MONTH_NAME, DAY_NAME, QUARTER, DEP_HOUR, TIME_OF_DAY |
| 2d | Category optimization | Converted 9 columns to category dtype |
| 3a | Outlier detection | IQR analysis on 7 numerical columns |
| 3b | Outlier flags | IS_EXTREME_DEP_DELAY, IS_EXTREME_ARR_DELAY, capped delay columns |
| 4a | Delay classification | DELAY_CATEGORY (5 buckets) |
| 4b | Delayed flag | IS_DELAYED (>15 min industry standard) |
| 4c | Delay breakdown | TOTAL_DELAY_BREAKDOWN, PRIMARY_DELAY_CAUSE |
| 4d | Distance categories | Short-haul, Medium-haul, Long-haul |
| 4e | Route column | ORIGIN -> DESTINATION |
| 5 | Merge references | Airline names, airport names/cities/states/coordinates |
| 6 | Validation | Sanity checks, row count verification |
| 7 | Export | Saved to data/processed/flights_cleaned.csv |

**Next Step:** Proceed to `03_eda.ipynb` for Exploratory Data Analysis.